# Züricom RAG Hands-on Workshop

## Exercise 1 — Campaign Launch
### Configure a RAG system for a real marketing task

## Exercise 2 — Competitive Positioning
### Troubleshoot a RAG system that is behaving strangely

You are working as a marketing team at **Züricom**, a fictional Swiss telecommunications company.

This notebook is deliberately **not a coding exercise**. The RAG pipeline is already implemented. Your job is to experiment with a small set of visible controls, inspect what the system retrieves, and reason about why the answer changes.

### The pipeline

```text
Marketing question
      ↓
   Chunking          ← configure
      ↓
   Embeddings
      ↓
    Chroma
      ↓
   Retrieval         ← configure
      ↓
 Retrieved context  ← inspect
      ↓
      LLM             ← configure
      ↓
     Answer
```

The most important habit is:

> **Do not debug only the final answer. Inspect what happened earlier in the pipeline.**

### How to use this notebook

1. Run the setup cells once.
2. In each blue **form**, change the controls using dropdowns/sliders instead of editing Python.
3. In Exercise 1, rerun the **controls + index** cell after changing a RAG parameter.
4. Edit the prompts in the clearly marked **Prompt editor** cell when you want to test prompt changes.
5. In Exercise 2, each incident has its own **form + index + answer** cell. Diagnose before changing anything.
6. Exact generated wording may differ between runs. Focus on the **evidence, failure pattern and effect of your change**, not on reproducing one exact sentence.


## Learning goals

By the end of the practical, you should be able to:

- explain what chunking changes in a RAG system;
- explain how `top_k`, similarity thresholds and metadata filters affect retrieval;
- inspect retrieved chunks and decide whether the system has the evidence it needs;
- distinguish a **retrieval problem** from a **generation problem**;
- recognize that retrieved information can still be outdated or conflicting;
- make a configuration change, rerun the system, and verify whether the change helped.

### What you do *not* need to learn

You do not need to understand vector-index internals or write a search algorithm. LangChain and Chroma handle that plumbing so that you can focus on RAG behaviour.


## 0. Setup

Run the next two cells once. The notebook automatically downloads the Züricom corpus from the workshop GitHub repository, so there is no manual file upload or ZIP handling.


In [ ]:
# Install the open-source packages used in the workshop. No API keys are required.
%pip install -qU \
    "langchain>=1,<2" \
    "langchain-chroma>=0.2,<1" \
    "langchain-huggingface>=0.3,<2" \
    "langchain-text-splitters>=1,<2" \
    "chromadb>=1,<2" \
    "sentence-transformers>=3,<6" \
    "transformers>=4.45,<5" \
    "accelerate>=1,<2"


In [ ]:
# Download the corpus from the public GitHub repository. You do not need to edit this cell.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/VJan-fin/HWZ-CAS-Advanced-AI-in-marketing.git"
REPO_DIR = "/content/HWZ-CAS-Advanced-AI-in-marketing"

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

CORPUS_DIR = repo_path / "corpus"
if not CORPUS_DIR.exists():
    raise FileNotFoundError(f"Expected corpus folder at {CORPUS_DIR}")

print(f"Corpus ready: {CORPUS_DIR}")


In [ ]:
# Imports and environment check. You do not need to edit this cell.
import re
import gc
import os
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import torch
from IPython.display import display, Markdown
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Python: {os.sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# Load the Züricom documents from the corpus folder. You do not need to edit this cell.

def parse_front_matter(text: str) -> Tuple[Dict[str, str], str]:
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", text, flags=re.DOTALL)
    if not match:
        return {}, text
    metadata = {}
    for line in match.group(1).splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip().strip('"').strip("'")
    return metadata, match.group(2)


documents: List[Document] = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name.startswith("00_") or "README" in path.name.upper() or "INSTRUCTOR" in path.name.upper():
        continue
    front_matter, body = parse_front_matter(path.read_text(encoding="utf-8"))
    front_matter["source_filename"] = path.name
    documents.append(Document(page_content=body.strip(), metadata=front_matter))

if not documents:
    raise RuntimeError("No Markdown source documents were found in the corpus folder.")

print(f"Loaded {len(documents)} Züricom source documents.")
print("Examples:", ", ".join(d.metadata["source_filename"] for d in documents[:4]))


### The Züricom knowledge base

The corpus contains product facts, audience personas, brand/channel guidance, customer research and dated competitor snapshots.

Some documents intentionally overlap. That is realistic and important: retrieval has to find the **right evidence**, not just any text that sounds related.


In [ ]:
# Fixed model choices for the workshop. Students experiment mainly with RAG parameters.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": embedding_device},
    encode_kwargs={"normalize_embeddings": True},
)

model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL,
    device_map="auto",
    torch_dtype=model_dtype,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Generation model: {GENERATION_MODEL}")
print(f"Generation device: {model.device}")


In [ ]:
# Chunking helpers — you do not need to edit this code.

def fixed_chunks(doc: Document) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=int(CHUNK_SIZE),
        chunk_overlap=int(CHUNK_OVERLAP),
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    return splitter.split_documents([doc])


def structure_chunks(doc: Document) -> List[Document]:
    sections = re.split(r"(?=^#{1,3}\s+)", doc.page_content, flags=re.MULTILINE)
    sections = [s.strip() for s in sections if s.strip()]
    fallback = RecursiveCharacterTextSplitter(
        chunk_size=int(CHUNK_SIZE),
        chunk_overlap=int(CHUNK_OVERLAP),
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    output = []
    for section in sections:
        if len(section) <= int(CHUNK_SIZE * 1.35):
            output.append(Document(page_content=section, metadata=dict(doc.metadata)))
        else:
            output.extend(
                fallback.split_documents(
                    [Document(page_content=section, metadata=dict(doc.metadata))]
                )
            )
    return output


def semantic_chunks(doc: Document) -> List[Document]:
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", doc.page_content) if p.strip()]
    if len(paragraphs) <= 1:
        return [Document(page_content=doc.page_content, metadata=dict(doc.metadata))]

    vectors = np.asarray(embeddings.embed_documents(paragraphs), dtype=float)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True).clip(min=1e-12)

    chunks_local = []
    current = paragraphs[0]
    for i in range(1, len(paragraphs)):
        similarity = float(np.dot(vectors[i-1], vectors[i]))
        next_text = paragraphs[i]
        should_break = (
            similarity < float(SEMANTIC_BREAK_THRESHOLD)
            or len(current) + len(next_text) + 2 > int(CHUNK_SIZE)
        )
        if should_break:
            chunks_local.append(current)
            current = next_text
        else:
            current += "\n\n" + next_text
    chunks_local.append(current)
    return [Document(page_content=c, metadata=dict(doc.metadata)) for c in chunks_local]


def build_chunks() -> List[Document]:
    strategy = CHUNKING_STRATEGY
    if strategy == "Fixed":
        fn = fixed_chunks
    elif strategy == "Structure-aware":
        fn = structure_chunks
    elif strategy == "Semantic-lite":
        fn = semantic_chunks
    else:
        raise ValueError(f"Unknown chunking strategy: {strategy}")

    all_chunks = []
    counters = {}
    for doc in documents:
        all_chunks.extend(fn(doc))
    for chunk in all_chunks:
        doc_id = chunk.metadata.get("doc_id", chunk.metadata.get("source_filename", "unknown"))
        idx = counters.get(doc_id, 0)
        counters[doc_id] = idx + 1
        chunk.metadata["chunk_index"] = idx
        chunk.metadata["chunk_id"] = f"{doc_id}::chunk_{idx:03d}"
    return all_chunks


# Exercise 1 — Campaign Launch

## Your challenge

Züricom wants to promote its **Unlimited Plan** to **frequent travelers in Switzerland**.

Configure the RAG system so that it can produce:

1. a campaign proposition;
2. three key messages;
3. an email subject line and short opening;
4. a short LinkedIn post.

The result should be:

- factually accurate;
- relevant to frequent travelers;
- consistent with Züricom's marketing style;
- careful with important qualifiers such as geography and eligibility;
- based on information from the knowledge base rather than invented claims.

There is **no single correct configuration**. Your goal is to find a configuration that works well and understand *why* it works.


## 1. Tune the RAG system with the form

The starting values are deliberately **suboptimal but plausible**. Change one or two things at a time and observe what happens.

**Rerun this one cell whenever you change a control.** It rebuilds the in-memory Chroma index using your new chunking settings.


In [ ]:
# @title 🎛️ Exercise 1 — RAG controls + index {display-mode: "form"}

CHUNKING_STRATEGY = "Fixed" # @param ["Fixed", "Structure-aware", "Semantic-lite"]
CHUNK_SIZE = 900 # @param {type:"slider", min:300, max:1400, step:50}
CHUNK_OVERLAP = 0 # @param {type:"slider", min:0, max:100, step:10}
SEMANTIC_BREAK_THRESHOLD = 0.75 # @param {type:"slider", min:0.50, max:0.90, step:0.05}

TOP_K = 1 # @param {type:"slider", min:1, max:8, step:1}
SIMILARITY_THRESHOLD = 0.00 # @param {type:"slider", min:0.00, max:0.90, step:0.05}
DOCUMENT_FILTER = "No metadata filter" # @param ["No metadata filter", "Current documents only"]

TEMPERATURE = 0.7 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

print("Current settings:")
print(f"  Chunking: {CHUNKING_STRATEGY} | size={CHUNK_SIZE} | overlap={CHUNK_OVERLAP}")
print(f"  Top K: {TOP_K} | relevance threshold: {SIMILARITY_THRESHOLD:.2f}")
print(f"  Metadata filter: {DOCUMENT_FILTER}")
print(f"  Temperature: {TEMPERATURE:.1f}")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("Chunk overlap must be smaller than chunk size.")

chunks = build_chunks()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"zuricom_workshop_{os.urandom(4).hex()}",
)

print(f"Indexed {len(chunks)} chunks using {CHUNKING_STRATEGY} chunking.")


In [ ]:
# Inspect what the current chunking strategy produced.
preview = []
for c in chunks[:18]:
    preview.append({
        "source": c.metadata.get("source_filename"),
        "chunk": c.metadata.get("chunk_index"),
        "chars": len(c.page_content),
        "preview": re.sub(r"\s+", " ", c.page_content)[:240] + ("…" if len(c.page_content) > 240 else ""),
    })
display(__import__("pandas").DataFrame(preview))


## 2. Prompt editor

The prompt below is a **starting point, not the solution**. Improve it during the exercise.

Keep both parts visible here so you can clearly see the instructions sent to the model:

- **System prompt:** defines the assistant's role and general rules.
- **User prompt template:** combines your instructions with the user's request and the retrieved context.

A `ChatPromptTemplate` keeps those pieces explicit and makes it easy to change the wording without changing the RAG pipeline itself. LangChain supports separate system/human messages in `ChatPromptTemplate`.


In [ ]:
# @title ✍️ Exercise 1 — Prompt editor

from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """You are Züricom's marketing assistant.
Answer the user's request using the retrieved context.
Do not invent product facts."""

USER_PROMPT_TEMPLATE = """Create useful marketing content for the request below.

USER REQUEST:
{question}

RETRIEVED CONTEXT:
{context}

Use the retrieved context when it is relevant."""

PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_PROMPT_TEMPLATE),
])

print("Prompt ready. Edit SYSTEM_PROMPT and USER_PROMPT_TEMPLATE, then rerun this cell before testing.")


In [ ]:
# Retrieval and generation helpers — you do not need to edit this code.

def metadata_filter_from_form():
    if DOCUMENT_FILTER == "Current documents only":
        return {"status": "current"}
    return None


def retrieve(query: str, top_k: Optional[int] = None, threshold: Optional[float] = None, metadata_filter=None):
    k = int(TOP_K if top_k is None else top_k)
    raw_threshold = SIMILARITY_THRESHOLD if threshold is None else float(threshold)
    filt = metadata_filter_from_form() if metadata_filter is None else metadata_filter
    results = vector_store.similarity_search_with_relevance_scores(query, k=k, filter=filt)
    if raw_threshold > 0:
        results = [(doc, score) for doc, score in results if float(score) >= raw_threshold]
    return results


def show_retrieval(query: str, show_full_text: bool = False):
    results = retrieve(query)
    rows = []
    for rank, (doc, score) in enumerate(results, start=1):
        rows.append({
            "rank": rank,
            "relevance": round(float(score), 3),
            "source": doc.metadata.get("source_filename"),
            "status": doc.metadata.get("status", ""),
            "version": doc.metadata.get("version", ""),
            "chunk": doc.metadata.get("chunk_index"),
            "text": doc.page_content if show_full_text else re.sub(r"\s+", " ", doc.page_content)[:320] + ("…" if len(doc.page_content) > 320 else ""),
        })
    if rows:
        display(__import__("pandas").DataFrame(rows))
    else:
        print("No chunks passed the current retrieval settings.")
    return results


def context_text(results) -> str:
    if not results:
        return "[NO RETRIEVED CONTEXT]"
    blocks = []
    for i, (doc, score) in enumerate(results, start=1):
        blocks.append(
            f"[Source {i} | {doc.metadata.get('source_filename')} | {doc.metadata.get('chunk_id')} | relevance={float(score):.3f}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(blocks)


def hf_messages(prompt_messages):
    role_map = {"system": "system", "human": "user", "ai": "assistant"}
    return [
        {"role": role_map.get(message.type, message.type), "content": message.content}
        for message in prompt_messages
    ]


def generate_answer(query: str, prompt_template: ChatPromptTemplate = None, show_context: bool = True, temperature: Optional[float] = None) -> str:
    results = retrieve(query)
    template = PROMPT_TEMPLATE if prompt_template is None else prompt_template
    prompt_messages = template.format_messages(question=query, context=context_text(results))
    messages = hf_messages(prompt_messages)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    temp = TEMPERATURE if temperature is None else float(temperature)
    kwargs = {
        "max_new_tokens": 220,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temp > 0:
        kwargs.update({"do_sample": True, "temperature": temp, "top_p": 0.9})
    else:
        kwargs.update({"do_sample": False})

    with torch.no_grad():
        output = model.generate(**inputs, **kwargs)
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if show_context:
        print("\n===== RETRIEVED EVIDENCE =====")
        show_retrieval(query, show_full_text=False)
        print("\n===== GENERATED ANSWER =====")
    return answer


In [ ]:
# @title ▶️ Run the campaign query
CAMPAIGN_QUERY = """Create a campaign for Züricom's Unlimited Plan aimed at frequent travelers in Switzerland.
Give me a campaign proposition, three key messages, an email subject line with a short opening,
and a short LinkedIn post. Emphasize the benefits most relevant to this audience."""

print(generate_answer(CAMPAIGN_QUERY))


## 3. Experiment deliberately

Try at least three configurations. Change **one meaningful thing at a time**.

Good questions to investigate:

- What changes when `TOP_K` goes from 1 → 3 → 5?
- Does structure-aware or semantic-lite chunking change which evidence is retrieved?
- Does overlap help preserve an important qualifier across a chunk boundary?
- Does a similarity threshold remove useful evidence or irrelevant evidence?
- **Does restricting the search based on metadata change the result?**
- How much can you improve the answer by changing the prompt rather than the retriever?
- Can you improve the answer without simply retrieving a large amount of context?

### Keep a small experiment log

| Experiment | What did you change? | What changed in retrieval? | What changed in the answer? | Why do you think it changed? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |


# Exercise 2 — Competitive Positioning: Troubleshooting Lab

The assistant is now in production. The marketing team reports three incidents.

**Each incident starts from a separate, deliberately broken configuration.** Do not simply carry your Exercise 1 settings into the cases.

Because the LLM is non-deterministic, the exact wording in your run may differ from the reported symptom. That is intentional. Focus on the **underlying failure pattern** and use the retrieved evidence to test your hypothesis.

For every case:

1. read the query and the reported symptom;
2. inspect the current parameters;
3. inspect the retrieved evidence;
4. identify the **first point in the pipeline where something may have gone wrong**;
5. make the smallest change you think is justified;
6. rerun and decide whether the change actually helped.

Use this mental model:

```text
Query → Chunking → Retrieval → Context → Generation → Answer
```

The key discipline is **diagnose first, change second**.


## Case 1 — The comparison is strangely generic

### Query

> **Compare Züricom's Unlimited Plan with HelvetiTel for frequent travelers. What are our strongest differentiators?**

### Reported symptom

The marketing team says the assistant often gives a broad, positive comparison but fails to surface one or more **concrete roaming differences**.

Your run may phrase the answer differently. Do not try to reproduce a sentence exactly.

### Your task

Inspect the retrieved chunks and relevance scores before changing anything.

Ask yourself:

- Is the Züricom product evidence there?
- Is the HelvetiTel evidence there?
- Is the specific roaming evidence there?
- How much evidence is reaching the model?
- What does that suggest about recall?


In [ ]:
# @title 🐛 Case 1 — Controls + index + answer {display-mode: "form"}
CHUNKING_STRATEGY = "Fixed" # @param ["Fixed", "Structure-aware", "Semantic-lite"]
CHUNK_SIZE = 900 # @param {type:"slider", min:300, max:1400, step:50}
CHUNK_OVERLAP = 0 # @param {type:"slider", min:0, max:100, step:10}
SEMANTIC_BREAK_THRESHOLD = 0.75 # @param {type:"slider", min:0.50, max:0.90, step:0.05}
TOP_K = 1 # @param {type:"slider", min:1, max:8, step:1}
SIMILARITY_THRESHOLD = 0.00 # @param {type:"slider", min:0.00, max:0.90, step:0.05}
DOCUMENT_FILTER = "No metadata filter" # @param ["No metadata filter", "Current documents only"]
TEMPERATURE = 0.2 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

CASE_1_SYSTEM_PROMPT = """You are Züricom's competitive positioning assistant.
Use the retrieved context to compare the two offers. Do not invent facts."""

CASE_1_USER_PROMPT = """Compare Züricom's Unlimited Plan with HelvetiTel for frequent travelers.
What are our strongest differentiators?

Use the retrieved context below as your evidence.

RETRIEVED CONTEXT:
{context}"""

CASE_1_PROMPT = ChatPromptTemplate.from_messages([
    ("system", CASE_1_SYSTEM_PROMPT),
    ("human", CASE_1_USER_PROMPT),
])

chunks = build_chunks()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"zuricom_case1_{os.urandom(4).hex()}",
)

CASE_1_QUERY = "Compare Züricom's Unlimited Plan with HelvetiTel for frequent travelers. What are our strongest differentiators?"
print("CURRENT SETTINGS")
print(f"  Chunking: {CHUNKING_STRATEGY} | size={CHUNK_SIZE} | overlap={CHUNK_OVERLAP}")
print(f"  Top K: {TOP_K} | threshold: {SIMILARITY_THRESHOLD:.2f} | metadata filter: {DOCUMENT_FILTER}")
print("\nRETRIEVED EVIDENCE")
show_retrieval(CASE_1_QUERY, show_full_text=True)
print("\nCURRENT ANSWER")
print(generate_answer(CASE_1_QUERY, prompt_template=CASE_1_PROMPT, show_context=False, temperature=TEMPERATURE))


### Case 1 — Diagnose

A likely hypothesis is a retrieval-recall problem, but **do not accept that explanation until the evidence supports it**.

Try the smallest change you think is justified. If you change chunking, this same cell rebuilds the index automatically.


## Case 2 — The competitor price is inconsistent

### Query

> **What is HelvetiTel's current monthly price, and how does it compare with Züricom?**

### Reported production incident

The marketing team reported an answer using **CHF 49/month** for HelvetiTel, even though a newer competitive snapshot says **CHF 59/month**.

### Your task

Inspect the retrieved evidence. The important question is not simply “why did the wrong chunk get retrieved?”

Ask yourself:

- Are both versions being retrieved?
- What metadata distinguishes them?
- Is semantic similarity enough to determine which version should be authoritative?
- What business rule could you express with metadata?


In [ ]:
# @title 🐛 Case 2 — Controls + index + answer {display-mode: "form"}
CHUNKING_STRATEGY = "Structure-aware" # @param ["Fixed", "Structure-aware", "Semantic-lite"]
CHUNK_SIZE = 1000 # @param {type:"slider", min:300, max:1400, step:50}
CHUNK_OVERLAP = 0 # @param {type:"slider", min:0, max:100, step:10}
SEMANTIC_BREAK_THRESHOLD = 0.75 # @param {type:"slider", min:0.50, max:0.90, step:0.05}
TOP_K = 6 # @param {type:"slider", min:1, max:8, step:1}
SIMILARITY_THRESHOLD = 0.00 # @param {type:"slider", min:0.00, max:0.90, step:0.05}
DOCUMENT_FILTER = "No metadata filter" # @param ["No metadata filter", "Current documents only"]
TEMPERATURE = 0.0 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

CASE_2_SYSTEM_PROMPT = """You are Züricom's competitive intelligence assistant.
Answer using the retrieved context. When sources conflict, be explicit about the evidence rather than inventing a resolution."""

CASE_2_USER_PROMPT = """What is HelvetiTel's current monthly price, and how does it compare with Züricom?

RETRIEVED CONTEXT:
{context}"""

CASE_2_PROMPT = ChatPromptTemplate.from_messages([
    ("system", CASE_2_SYSTEM_PROMPT),
    ("human", CASE_2_USER_PROMPT),
])

chunks = build_chunks()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"zuricom_case2_{os.urandom(4).hex()}",
)

CASE_2_QUERY = "What is HelvetiTel's current monthly price, and how does it compare with Züricom?"
print("CURRENT SETTINGS")
print(f"  Chunking: {CHUNKING_STRATEGY} | size={CHUNK_SIZE} | overlap={CHUNK_OVERLAP}")
print(f"  Top K: {TOP_K} | threshold: {SIMILARITY_THRESHOLD:.2f} | metadata filter: {DOCUMENT_FILTER}")
print("\nRETRIEVED EVIDENCE")
show_retrieval(CASE_2_QUERY, show_full_text=True)
print("\nCURRENT ANSWER")
print(generate_answer(CASE_2_QUERY, prompt_template=CASE_2_PROMPT, show_context=False, temperature=TEMPERATURE))


### Case 2 — Diagnose

Look carefully at the two HelvetiTel snapshots.

A useful experiment is to change the **metadata filter**, then compare the retrieved evidence before and after.

The point is not that “current documents only” is universally correct. The point is to recognize that **semantic relevance and business authority are different questions**.


## Case 3 — The evidence is right, but the claim is wrong

### Query

> **Write a one-sentence positioning statement highlighting Züricom's EU roaming advantage over HelvetiTel for frequent travelers.**

### Reported production incident

A production response was reported as:

> **“Roam freely throughout Europe with Züricom's unlimited data, calls and SMS.”**

Your local generation may not reproduce that sentence exactly. The question is whether the **same kind of problem** occurs.

### Your task

Inspect the retrieved evidence and the prompt before changing anything.

Ask yourself:

- Does the retrieved context contain the information needed to answer the question?
- Does it contain the exact geographic qualifier?
- If the evidence is correct, where else could the problem have started?
- Can the prompt be made more explicit about preserving qualifiers?


In [ ]:
# @title 🐛 Case 3 — Controls + index + answer {display-mode: "form"}
CHUNKING_STRATEGY = "Structure-aware" # @param ["Fixed", "Structure-aware", "Semantic-lite"]
CHUNK_SIZE = 1000 # @param {type:"slider", min:300, max:1400, step:50}
CHUNK_OVERLAP = 0 # @param {type:"slider", min:0, max:100, step:10}
SEMANTIC_BREAK_THRESHOLD = 0.75 # @param {type:"slider", min:0.50, max:0.90, step:0.05}
TOP_K = 4 # @param {type:"slider", min:1, max:8, step:1}
SIMILARITY_THRESHOLD = 0.00 # @param {type:"slider", min:0.00, max:0.90, step:0.05}
DOCUMENT_FILTER = "No metadata filter" # @param ["No metadata filter", "Current documents only"]
TEMPERATURE = 0.2 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

CASE_3_SYSTEM_PROMPT = """You are Züricom's marketing assistant.
Write persuasive marketing copy using the retrieved context. Keep it concise."""

CASE_3_USER_PROMPT = """Write a one-sentence positioning statement highlighting Züricom's EU roaming advantage over HelvetiTel for frequent travelers.

RETRIEVED CONTEXT:
{context}"""

CASE_3_PROMPT = ChatPromptTemplate.from_messages([
    ("system", CASE_3_SYSTEM_PROMPT),
    ("human", CASE_3_USER_PROMPT),
])

chunks = build_chunks()
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f"zuricom_case3_{os.urandom(4).hex()}",
)

CASE_3_QUERY = "Write a one-sentence positioning statement highlighting Züricom's EU roaming advantage over HelvetiTel for frequent travelers."
print("CURRENT SETTINGS")
print(f"  Chunking: {CHUNKING_STRATEGY} | size={CHUNK_SIZE} | overlap={CHUNK_OVERLAP}")
print(f"  Top K: {TOP_K} | threshold: {SIMILARITY_THRESHOLD:.2f} | metadata filter: {DOCUMENT_FILTER}")
print("\nRETRIEVED EVIDENCE")
show_retrieval(CASE_3_QUERY, show_full_text=True)
print("\nCURRENT ANSWER")
print(generate_answer(CASE_3_QUERY, prompt_template=CASE_3_PROMPT, show_context=False, temperature=TEMPERATURE))


### Case 3 — Diagnose and fix

If the retrieved evidence contains the correct roaming rule and its exact geographic qualifier, retrieval may not be the first failure point.

Try a prompt change that explicitly tells the model to preserve **scope, eligibility, price and other important qualifiers**. Then rerun the cell and compare the answer with the evidence.

The goal is not to discover a magic prompt. The goal is to recognize when the **right context is already present** and the next place to investigate is generation.


# Final reflection

Answer these in your own words before moving on:

1. If the retrieved chunks do not contain the information needed for an answer, can a better prompt fix the problem?
2. If the retrieved chunks contain two conflicting versions of a fact, is the problem necessarily the embedding model?
3. If the right chunks are retrieved but the answer changes an important qualifier, where would you investigate first?
4. Which parameters mattered most for the campaign task?
5. What would you want to measure systematically before declaring this RAG system “good enough”?

## Takeaway

> **A RAG system is a pipeline, not just an LLM.**
>
> When something goes wrong, trace the problem from **query → retrieval → context → generation → answer**.
